# Đấu vòng tròn giữa các tác tử — CHẠY

Notebook này **chỉ chạy đấu và lưu kết quả**. Xem và vẽ biểu đồ ở `arena_analysis.ipynb` (đọc `runs/arena_matrix/games.csv`),
nên đổi cách xem không cần đánh lại.

Mỗi **cặp** tác tử đấu `N_GAMES` ván (mặc định 100), **chia đều đi trước**: cùng một khai cuộc đánh 2 lần, đổi bên X/O
(50 khai cuộc × 2 bên). Khai cuộc là 2–4 nước ngẫu nhiên ở vùng giữa bàn (`runs/openings/openings_v1.json`).
Không đấu từ bàn trống: hai tác tử cấp 5 với `temperature: 0` là tất định, bàn trống chỉ cho đúng 1 ván lặp lại.

Chỉ chạy lại notebook này khi đổi tác tử, ngân sách hoặc số ván.

In [ ]:
import os
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
os.chdir(ROOT)   # mọi đường dẫn tương đối tính từ gốc dự án

import itertools, json, time

import pandas as pd

from caro.agents import Limits, make_agent
from caro.arena.match import play_match
from caro.arena.openings import load_openings

## 1. Cấu hình

Sửa `AGENTS` để thêm/bớt tác tử; `limits` đặt ngân sách mỗi nước (cấp 3 và 4 không có giới hạn mặc định nên rất chậm nếu bỏ trống).

In [ ]:
N_GAMES = 100          # số ván mỗi cặp (chẵn: một nửa mỗi bên đi trước)
SEED = 2026
OUT_DIR = Path("runs/arena_matrix")

# tên hiển thị -> (file cấu hình, ngân sách mỗi nước)
AGENTS = {
    "L1 random":  ("agents/l1_random",    None),
    "L2 rule":    ("agents/l2_rule",      None),
    "L3 AB d3":   ("agents/l3_alphabeta", Limits(max_depth=3)),
    "L4 MCTS200": ("agents/l4_mcts",      Limits(max_sims=200)),
    "L5 v1":      ("agents/l5_local_v1",  None),
    "L5 v2":      ("agents/l5_local_v2",  None),
}

assert N_GAMES % 2 == 0
agents = {}
for name, (cfg, limits) in AGENTS.items():
    a = make_agent(cfg)
    if limits is not None:
        a.limits = limits
    agents[name] = a
names = list(agents)
print("Tác tử:", names)

## 2. Khai cuộc

In [ ]:
openings = load_openings()
n_open = N_GAMES // 2
assert len(openings) >= n_open, f"chỉ có {len(openings)} khai cuộc"
openings = openings[:n_open]
print(f"{n_open} khai cuộc x 2 bên = {N_GAMES} ván / cặp;", "ví dụ:", openings[:3])

## 3. Chạy đấu

Mỗi cặp (A, B): với mỗi khai cuộc, đánh một ván A đi trước (X) và một ván B đi trước. Chạy tuần tự, có in tiến độ.

In [ ]:
rows, game_id = [], 0
pairs = list(itertools.combinations(names, 2))
t0 = time.time()
for p, (A, B) in enumerate(pairs, 1):
    tp = time.time()
    for k, opening in enumerate(openings):
        for first, second in ((A, B), (B, A)):
            r = play_match(agents[first], agents[second], opening, SEED, game_id)
            rows.append({"pair": f"{A} vs {B}", "A": A, "B": B, "opening_id": k,
                         "agent_x": first, "agent_o": second,
                         "result": r["result"], "plies": len(r["moves"]),
                         "termination": r["termination"]})
            game_id += 1
    print(f"[{p}/{len(pairs)}] {A} vs {B}: {time.time()-tp:.0f}s")
games = pd.DataFrame(rows)
print(f"Xong {len(games)} ván trong {time.time()-t0:.0f}s")

OUT_DIR.mkdir(parents=True, exist_ok=True)
games.to_csv(OUT_DIR / "games.csv", index=False)

# metadata để notebook phân tích biết thứ tự tác tử và cấu hình đã dùng
meta = {"agents": {n: {"config": cfg, "limits": None if lim is None else vars(lim)}
                   for n, (cfg, lim) in AGENTS.items()},
        "n_games": N_GAMES, "seed": SEED, "n_openings": len(openings)}
(OUT_DIR / "meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print("Đã lưu", OUT_DIR / "games.csv", "-> mở arena_analysis.ipynb để xem kết quả")